# Did solubility cause the 5G51 expression failures?

Scores all 15 panel designs with **soluble ProteinMPNN** vs **vanilla ProteinMPNN**, then tests
whether that score separates the ones that expressed from the ones that didn't.

**Two scoring contexts, and the distinction matters:**

- **monomer** — binder chain alone. This is the state during expression in *E. coli*; there is no
  target in the cell. **This is the one relevant to the expression hypothesis.**
- **complex** — binder + 5G51 target. Relevant to binding, not to expression. Included as a
  contrast: interface residues look "buried and fine" in complex but are exposed hydrophobic
  patches on the lone binder, which is exactly how a good binder can be a bad protein.

ProteinMPNN `--score_only` returns the **mean negative log-likelihood** of the actual sequence
given the backbone. **Lower = the model likes it more.** The informative quantity is
`delta = soluble − vanilla`: how much *worse* the soluble-trained model rates the sequence than
the general model. A large positive delta means "this looks like a protein, but not a soluble one."

CPU is fine — 15 designs, no GPU needed.

## 1 · Upload the panel bundle

In [ ]:
# Upload exports/5g51_panel15_solubility.zip when prompted.
from google.colab import files
import zipfile, os, glob

# Upload one or BOTH bundles:
#   5g51_panel15_solubility.zip          - the 15 that went to Adaptyv (have expression data)
#   5g51_bindcraft_aridge8_solubility.zip - the 8 new BindCraft A_ridge accepts
up = files.upload()
for zname in up:
    with zipfile.ZipFile(zname) as z:
        z.extractall("/content")

PANELS = sorted(os.path.dirname(m) for m in glob.glob("/content/*/manifest.csv"))
assert PANELS, "no panel found - did the zip extract?"
for P in PANELS:
    print(f"{os.path.basename(P):40s} monomer={len(glob.glob(P+'/monomer/*.pdb')):3d} "
          f"complex={len(glob.glob(P+'/complex/*.pdb')):3d}")

## 2 · Install ProteinMPNN (needs only torch + numpy, both preinstalled)

In [ ]:
import subprocess, os, torch
if not os.path.isdir("/content/ProteinMPNN"):
    subprocess.run("git clone -q --depth 1 https://github.com/dauparas/ProteinMPNN.git "
                   "/content/ProteinMPNN", shell=True, check=True)
print("torch", torch.__version__)
print("soluble weights:", os.listdir("/content/ProteinMPNN/soluble_model_weights"))
print("vanilla weights:", os.listdir("/content/ProteinMPNN/vanilla_model_weights"))

## 3 · Score

`v_48_020` in both cases (48 neighbours, 0.20 Å backbone noise) so soluble vs vanilla is the
*only* difference. `--pdb_path_chains A` restricts scoring to the binder chain, which matters
for the complex run — otherwise the 157-residue target dominates the average.
`--seed 37` fixes the decoding order so the comparison is deterministic.

In [ ]:
import subprocess, glob, os, numpy as np, re

MPNN = "/content/ProteinMPNN"

def score_dir(pdb_dir, soluble, tag):
    """Returns {design_name: mean negative log-likelihood of its own sequence}."""
    out = f"/content/scores_{tag}"
    os.makedirs(out, exist_ok=True)
    cmd = (f"cd {MPNN} && python protein_mpnn_run.py "
           f"--pdb_path {{pdb}} --pdb_path_chains A "
           f"--out_folder {out} --score_only 1 --seed 37 --batch_size 1 "
           f"--model_name v_48_020" + (" --use_soluble_model" if soluble else ""))
    res = {}
    for pdb in sorted(glob.glob(f"{pdb_dir}/*.pdb")):
        name = os.path.basename(pdb)[:-4]
        r = subprocess.run(cmd.format(pdb=pdb), shell=True, text=True, capture_output=True)
        npz = f"{out}/score_only/{name}_pdb.npz"
        if not os.path.exists(npz):
            print(f"  !! {name} failed\n{r.stderr[-600:]}")
            continue
        res[name] = float(np.load(npz)["score"].mean())
    print(f"{tag}: scored {len(res)}")
    return res

SCORES = {}   # SCORES[panel][key][design] = score
for P in PANELS:
    pn = os.path.basename(P)
    SCORES[pn] = {
        "sol_monomer": score_dir(f"{P}/monomer", True,  f"{pn}_sol_mono"),
        "van_monomer": score_dir(f"{P}/monomer", False, f"{pn}_van_mono"),
        "sol_complex": score_dir(f"{P}/complex", True,  f"{pn}_sol_cplx"),
        "van_complex": score_dir(f"{P}/complex", False, f"{pn}_van_cplx"),
    }

## 4 · Sequence-level solubility descriptors

MPNN is structure-conditioned, so it can miss purely compositional problems. These are the
classic sequence-only predictors of poor expression, computed independently as a cross-check.

In [ ]:
KD = {"A":1.8,"R":-4.5,"N":-3.5,"D":-3.5,"C":2.5,"Q":-3.5,"E":-3.5,"G":-0.4,
      "H":-3.2,"I":4.5,"L":3.8,"K":-3.9,"M":1.9,"F":2.8,"P":-1.6,"S":-0.8,
      "T":-0.7,"W":-0.9,"Y":-1.3,"V":4.2}

def descriptors(seq):
    n = len(seq)
    gravy = sum(KD[a] for a in seq) / n
    pos, neg = seq.count("K") + seq.count("R"), seq.count("D") + seq.count("E")
    hyd = sum(seq.count(a) for a in "AVILMFWY")
    # longest run of consecutive hydrophobics - aggregation-prone stretch
    run = best = 0
    for a in seq:
        run = run + 1 if a in "AVILMFWY" else 0
        best = max(best, run)
    return {"length": n, "gravy": round(gravy, 3),
            "net_charge": pos - neg, "charge_per_res": round((pos - neg) / n, 3),
            "pct_hydrophobic": round(100 * hyd / n, 1), "max_hydrophobic_run": best,
            "n_Cys": seq.count("C"), "n_Trp": seq.count("W"), "n_Met": seq.count("M")}
print("descriptors ready")

## 5 · Assemble the table

In [ ]:
import csv, os, pandas as pd

rows = []
for P in PANELS:
    pn = os.path.basename(P)
    for r in csv.DictReader(open(f"{P}/manifest.csv")):
        n = r["name"]; r["panel"] = pn
        r.update(descriptors(r["sequence"]))
        for k in ("sol_monomer","van_monomer","sol_complex","van_complex"):
            r[k] = round(SCORES[pn][k].get(n, float("nan")), 4)
        r["delta_monomer"] = round(r["sol_monomer"] - r["van_monomer"], 4)
        r["delta_complex"] = round(r["sol_complex"] - r["van_complex"], 4)
        rows.append(r)

df = pd.DataFrame(rows)
cols = ["panel","name","length","expressed","sol_monomer","van_monomer","delta_monomer",
        "sol_complex","van_complex","delta_complex","gravy","net_charge","pct_hydrophobic",
        "max_hydrophobic_run","n_Cys","n_Trp"]
df[cols].to_csv("/content/panel15_solubility_scores.csv", index=False)
print(df[cols].sort_values("sol_monomer").to_string(index=False))
print("\nsaved /content/panel15_solubility_scores.csv")

## 6 · Enter the expression results

`finalsub1` has no expression column, so fill it in here: `1` = expressed, `0` = did not.
Names must match the manifest exactly (see the table above).

In [ ]:
EXPRESSED = {
    # "5G51_V4_seed01": 1,
    # "5G51_orig_F_s8": 0,
    # ... fill in all 15
}

if not EXPRESSED:
    print("Fill in EXPRESSED above, then re-run this cell and cell 7.")
else:
    df["expressed"] = df["name"].map(EXPRESSED)
    missing = df[df["expressed"].isna()]["name"].tolist()
    if missing:
        print("!! no label for:", missing)
    print(df[["name","expressed","sol_monomer","delta_monomer","gravy"]].to_string(index=False))
    df.to_csv("/content/panel15_solubility_scores.csv", index=False)

## 7 · Does the score actually separate them?

Mann–Whitney U (rank-based, no normality assumption) plus the group medians. With n=15 split
maybe 4/11, this is badly underpowered — read effect sizes and the AUC, not the p-value.
An AUC near 0.5 means the metric carries no signal.

In [ ]:
import numpy as np
from scipy.stats import mannwhitneyu

if df["expressed"].notna().sum() < 4:
    print("Need the expression labels in cell 6 first.")
else:
    d = df.dropna(subset=["expressed"]).copy()
    d["expressed"] = d["expressed"].astype(int)
    yes, no = d[d.expressed == 1], d[d.expressed == 0]
    print(f"expressed n={len(yes)}   failed n={len(no)}\n")

    metrics = ["sol_monomer","delta_monomer","sol_complex","delta_complex",
               "gravy","net_charge","pct_hydrophobic","max_hydrophobic_run","n_Trp","length"]
    print(f"{'metric':22s} {'median(exp)':>12s} {'median(fail)':>13s} {'AUC':>6s} {'p':>8s}")
    print("-" * 66)
    for m in metrics:
        a, b = yes[m].astype(float), no[m].astype(float)
        if a.isna().any() or b.isna().any() or len(a) == 0 or len(b) == 0:
            continue
        u, p = mannwhitneyu(a, b, alternative="two-sided")
        auc = u / (len(a) * len(b))          # P(expressed ranks higher)
        flag = "  <-- separates" if (auc > 0.75 or auc < 0.25) else ""
        print(f"{m:22s} {a.median():12.3f} {b.median():13.3f} {auc:6.2f} {p:8.3f}{flag}")

    print("\nAUC 0.5 = no signal. AUC far from 0.5 = the metric tracks expression.")
    print("With this n, treat anything with p>0.05 but AUC>0.8 as a lead, not a result.")

## Interpreting it

**If `sol_monomer` separates the groups** (AUC well above 0.75 or below 0.25), solubility is a
plausible driver and soluble-MPNN is a usable pre-order filter for the A_ridge panel.

**If it doesn't**, the failures were probably not solubility — more likely the construct, codon
usage, tags, or the expression protocol itself. That would be a genuinely useful negative
result: it means re-ranking future designs on soluble-MPNN would be wasted effort.

**Either way, n=15 cannot settle this.** A four-vs-eleven split has very wide confidence
intervals, and with 10 metrics tested one will look significant by chance. Treat this as
hypothesis generation for the next panel, not proof.

Worth noting: the whole panel is 80–150 aa de novo helical bundles from the same pipeline, so
they share failure modes. If they clustered on some property, that property is confounded with
"came from this pipeline" and won't generalise.